# Model 2: LightGBM with Ratio Features
This notebook builds an advanced LightGBM model.
We also engineer ratio features to respect time relationships.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from lightgbm import LGBMClassifier
import sklearn
import os

sklearn.set_config(transform_output="pandas")

In [ ]:
# Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
# Separate features and target
X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [ ]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['number']).columns.tolist()

In [ ]:
# 1. Imputation
imputer = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numerical_cols),
        ('cat', SimpleImputer(strategy='most_frequent'), categorical_cols)
    ],
    verbose_feature_names_out=False
)

In [ ]:
# 2. Feature Engineering
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_out = X.copy()
        
        # Prevent division by zero
        denom = X_out['daily_screen_time_hours'].replace(0, 0.001)
        
        X_out['social_media_ratio'] = X_out['social_media_hours'] / denom
        X_out['gaming_ratio'] = X_out['gaming_hours'] / denom
        X_out['work_study_ratio'] = X_out['work_study_hours'] / denom
        
        return X_out

In [ ]:
# 3. Final Preprocessing
final_preprocessor = ColumnTransformer(
    transformers=[
        ('cat_encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

In [ ]:
# 4. Model Definition
model = LGBMClassifier(random_state=42, n_estimators=300, learning_rate=0.05)

In [ ]:
# Create and evaluate the pipeline
clf = Pipeline(steps=[
    ('imputer', imputer),
    ('engineer', FeatureEngineer()),
    ('final_preprocessor', final_preprocessor),
    ('model', model)
])

# Train the model
clf.fit(X, y)
print("LightGBM Model trained.")

In [ ]:
# Make predictions on test set
preds = clf.predict_proba(X_test)[:, 1]

In [ ]:
# Create submission file
os.makedirs('submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': preds})
submission.to_csv('submissions/submission_2.csv', index=False)
print("Submission saved to submissions/submission_2.csv")